# MDR-TS v13.2
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v13.2
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/base/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

Added a recursive temporal modeling experiment where the model now uses the previous timestep’s soil moisture prediction (t−1) as an input to predict the current timestep (t). This allows us to test whether explicit autoregressive structure improves performance beyond purely satellite and weather features. The implementation supports both teacher-forced training and fully recursive rollout at inference to evaluate real-world stability and error propagation.

## 0. Imports

In [11]:
import os
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

from xgboost import XGBRegressor

import torch

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

imports loaded
using: cpu


## 1. Environment Setup

In [12]:
# Reproducibility
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

# Environment / Runtime Info
def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    # Colab-specific checks
    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

# Plotting defaults
plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 2.1.2
  Running in Colab: False
  GPU available: False
environment setup complete


## 2. Data Access

In [13]:
# Project paths
VERSION = "v13.2"
SUBVERSION = "v13.2"
RUN_NAME = "mdr_ts_v13_2"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

# Create output directory if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v13.2/v13.2

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


## 3. Data Loading

In [14]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_new_updated/train_derived_new_updated.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_new_updated/val_derived_new_updated.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_new_updated/test_derived_new_updated.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new_updated/train_derived_new_updated.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new_updated/val_derived_new_updated.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new_updated/test_derived_new_updated.csv


In [15]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 356

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'elev', 'slope', 'aspect', 'DOY', 'soil_moisture_5cm', 'D_sin_DOY', 'D_cos_DOY', 'F_NDVI', 'F_NDMI', 'F_MSI', 'E_SAR_ratio', 'E_SAR_diff', 'G_API', 'G_DSLR', 'G_rain_sum_3d', 'G_rain_sum_7d', 'G_rain_sum_30d', 'A_d_G_API_kobs1']


## 4. Sanity Checks

In [16]:
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
  "precip_mm",

  "G_rain_sum_3d",
  "G_rain_sum_7d",
  "G_rain_sum_30d",
  "G_API",

  "G_DSLR",

  "C_lag_E_SAR_diff_kobs12",
  "C_lag_E_SAR_diff_kobs30",
  "C_lag_E_SAR_ratio_kobs30",
  "C_lag_F_NDVI_kobs30",
  "C_lag_LST_modis_kobs12",
  "C_lag_LST_modis_kobs30",
  "DOY",
  "D_sa_E_SAR_ratio",
  "D_sa_F_NDMI",
  "D_z_E_SAR_ratio",
  "D_z_F_NDMI",
  "E_SAR_diff",
  "E_SAR_ratio",
  "F_MSI",
  "F_NDMI",
  "V_ema_LST_modis_kobs30",
  "V_rollmax_E_SAR_diff_kobs14",
  "V_rollmax_E_SAR_diff_kobs30",
  "V_rollmax_F_NDVI_kobs30",
  "V_rollmax_G_API_kobs30",
  "V_rollmax_G_API_kobs7",
  "V_rollmax_LST_modis_kobs7",
  "V_rollmax_s2_b11_kobs30",
  "V_rollmean_G_API_kobs30",
  "V_rollmin_E_SAR_diff_kobs30",
  "V_rollmin_E_SAR_ratio_kobs30",
  "V_rollmin_F_NDMI_kobs30",
  "V_rollmin_G_API_kobs7",
  "V_rollmin_s2_b11_kobs30",
  "V_rollmin_s2_b12_kobs30",
  "s1_vh",
  "s2_b8",
  "A_d_LST_modis_kobs7",
  "A_grad_LST_modis_kobs14",
  "C_lag_E_SAR_diff_kobs6",
  "V_rollmax_E_SAR_diff_kobs7",
  "V_rollmax_E_SAR_ratio_kobs14",
  "V_rollmax_F_NDVI_kobs14",
  "s2_b12",
  "D_sa_LST_modis"
]

STATE_FEATURE_COLS = [
    "S_bucket",          # hidden wetness state s_t
    "S_bucket_d1",       # delta_s,t (s_t - s_{t-1})
    "S_bucket_ema7",     # short memory summary of s_t
    "S_bucket_ema30",    # longer memory summary of s_t
]

In [17]:
for d in (train_df, val_df, test_df):
    d["date"] = pd.to_datetime(d["date"], errors="coerce")

stations = sorted(set(train_df["station_id"].dropna().unique())
                  | set(val_df["station_id"].dropna().unique())
                  | set(test_df["station_id"].dropna().unique()))

print("\n=== TEMPORAL LEAKAGE CHECKS (per station) ===")
bad = 0

for sid in stations:
    tr = train_df[train_df["station_id"] == sid]["date"].dropna()
    va = val_df[val_df["station_id"] == sid]["date"].dropna()
    te = test_df[test_df["station_id"] == sid]["date"].dropna()

    if len(tr) == 0 or len(va) == 0 or len(te) == 0:
        print(f"[WARN] station {sid}: missing split data (train={len(tr)}, val={len(va)}, test={len(te)})")
        bad += 1
        continue

    tr_min, tr_max = tr.min(), tr.max()
    va_min, va_max = va.min(), va.max()
    te_min, te_max = te.min(), te.max()

    # date overlap checks (hard leakage)
    overlap_tr_va = len(set(tr.unique()) & set(va.unique()))
    overlap_tr_te = len(set(tr.unique()) & set(te.unique()))
    overlap_va_te = len(set(va.unique()) & set(te.unique()))

    # ordering check (soft but important)
    order_ok = (tr_max < va_min) and (va_max < te_min)

    if overlap_tr_va or overlap_tr_te or overlap_va_te or (not order_ok):
        print(f"[ERROR] station {sid}:")
        print(f"  train: {tr_min} -> {tr_max}")
        print(f"  val:   {va_min} -> {va_max}")
        print(f"  test:  {te_min} -> {te_max}")
        print(f"  overlaps: train∩val={overlap_tr_va}, train∩test={overlap_tr_te}, val∩test={overlap_va_te}")
        print(f"  order_ok: {order_ok}")
        bad += 1
    else:
        print(f"[OK] station {sid}: train<{val_df is not None and 'val' or ''}val<test with no date overlap")

if bad == 0:
    print("\n[INFO] No temporal leakage detected.")
else:
    print(f"\n[WARNING] {bad} station(s) have temporal leakage or split issues.")



=== TEMPORAL LEAKAGE CHECKS (per station) ===
[OK] station Darrington: train<valval<test with no date overlap
[OK] station Quinault: train<valval<test with no date overlap
[OK] station SourdoughGulch_WA_985: train<valval<test with no date overlap
[OK] station Spokane: train<valval<test with no date overlap
[WARN] station Touchet_WA_824: missing split data (train=3311, val=0, test=170)

[WARNING] 1 station(s) have temporal leakage or split issues.


## Autoregressive Rollout Experiment (P_lag1)
This experiment tests a recursive setup: predict soil moisture at time *t* using the previous timestep’s **prediction** as an additional input. Training uses teacher forcing (lagged true values), while validation/test use a strict per-station rollout (lagged predictions).


In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET_COL = "soil_moisture_5cm"
STATION_COL = "station_id"
DATE_COL = "date"

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def metrics_dict(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "R2": float(r2_score(y_true, y_pred)),
        "bias_mean(true-pred)": float(np.mean(y_true - y_pred)),
        "n": int(len(y_true)),
    }


### 1) Choose baseline feature columns
Use existing FEATURE_COLS if available; otherwise fall back to BASE_FEATURE_COLS. Then define AR feature columns by appending P_lag1 and P_lag1_missing.


In [19]:
if "FEATURE_COLS" in globals():
    BASE_FEATS = list(FEATURE_COLS)
elif "BASE_FEATURE_COLS" in globals():
    BASE_FEATS = list(BASE_FEATURE_COLS)
else:
    raise ValueError("Could not find FEATURE_COLS or BASE_FEATURE_COLS in notebook globals.")

AR_EXTRA_FEATS = ["P_lag1", "P_lag1_missing"]
AR_FEATS = BASE_FEATS + AR_EXTRA_FEATS

for c in [STATION_COL, DATE_COL, TARGET_COL]:
    if c not in train_df.columns:
        raise ValueError(f"Missing required column in train_df: {c}")

print("Base features:", len(BASE_FEATS))
print("AR features:  ", len(AR_FEATS))


Base features: 46
AR features:   48


### 2) Helper: build teacher-forced TRAIN matrix
P_lag1 = previous true soil moisture within each station (shift(1)). First row per station gets P_lag1=0 and missing flag=1.


In [20]:
def make_teacher_forced(df):
    df0 = df.copy()
    df0[DATE_COL] = pd.to_datetime(df0[DATE_COL], errors="coerce")
    df0 = df0.sort_values([STATION_COL, DATE_COL]).reset_index(drop=True)

    lag = df0.groupby(STATION_COL)[TARGET_COL].shift(1)
    miss = lag.isna().astype(np.int64)
    df0["P_lag1"] = lag.fillna(0.0).astype(float)
    df0["P_lag1_missing"] = miss

    X = df0[AR_FEATS]
    y = df0[TARGET_COL].astype(float)
    return df0, X, y

train_tf_df, X_train_ar, y_train_ar = make_teacher_forced(train_df)
print("Teacher-forced TRAIN shapes:", X_train_ar.shape, y_train_ar.shape)


Teacher-forced TRAIN shapes: (16972, 48) (16972,)


### 3) Base AR Model


In [26]:
# from v13.1
BASE_EST = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
    subsample=0.6,
    colsample_bytree=0.6,
    reg_lambda=5.0,
    reg_alpha=0.01,
    n_estimators=8000,
    min_child_weight=5,
    max_depth=6,
    learning_rate=0.01,
    gamma=0.0,
)

print("AR model configured from best XGB params")

AR model configured from best XGB params


### 4) Fit AR model on TRAIN (teacher forcing)
Fit on X_train_ar, y_train_ar. No early stopping / callbacks.


In [27]:
BASE_EST.fit(train_df[BASE_FEATS], train_df[TARGET_COL])
print("AR model fit complete.")


AR model fit complete.


In [28]:
yhat_train_base = BASE_EST.predict(train_df[BASE_FEATS]).ravel()
yhat_val_base   = BASE_EST.predict(val_df[BASE_FEATS]).ravel()
yhat_test_base  = BASE_EST.predict(test_df[BASE_FEATS]).ravel()

print("Baseline predictions ready.")
print("  train:", yhat_train_base.shape)
print("  val:  ", yhat_val_base.shape)
print("  test: ", yhat_test_base.shape)

Baseline predictions ready.
  train: (16972,)
  val:   (2919,)
  test:  (2829,)


### 5) Helper: recursive rollout prediction (VAL/TEST)
For each station, walk forward in time. At each row, set P_lag1 to the previous **prediction** for that station.
First row per station uses P_lag1=0 and P_lag1_missing=1.


In [32]:
def rollout_predict(df, fitted_model):
    df_in = df.copy()
    df_in[DATE_COL] = pd.to_datetime(df_in[DATE_COL], errors="coerce")

    # Coerce feature columns once
    for c in BASE_FEATS:
        if df_in[c].dtype == "object":
            df_in[c] = pd.to_numeric(df_in[c], errors="coerce")
    df_in[BASE_FEATS] = df_in[BASE_FEATS].astype(float).fillna(0.0)

    orig_idx = df_in.index.to_numpy()
    df_sorted = df_in.sort_values([STATION_COL, DATE_COL]).copy()

    yhat_sorted = np.zeros(len(df_sorted), dtype=float)

    for sid, g in df_sorted.groupby(STATION_COL, sort=False):
        idx = g.index.to_numpy()
        prev = 0.0
        first = True

        for j in idx:
            row = df_sorted.loc[j, BASE_FEATS].to_frame().T

            row["P_lag1"] = float(prev)
            row["P_lag1_missing"] = 1 if first else 0

            # ensure numeric
            row = row[AR_FEATS].astype(float).fillna(0.0)

            pred = float(fitted_model.predict(row)[0])
            yhat_sorted[df_sorted.index.get_loc(j)] = pred

            prev = pred
            first = False

    yhat_series = pd.Series(yhat_sorted, index=df_sorted.index).reindex(orig_idx)
    return yhat_series.to_numpy()

### 6) Baseline predictions (non-recursive) for comparison
Uses the existing baseline estimator (BASE_EST) and the base feature matrix (no P_lag1). If baseline is a stack/meta model already, use its existing predictions if available; otherwise predict directly on BASE_FEATS.


In [34]:
def baseline_predict(df, fitted_baseline, split_name=None):
    X = df[BASE_FEATS].copy()

    # ensure numeric for XGB
    for c in X.columns:
        if X[c].dtype == "object":
            X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.astype(float).fillna(0.0)

    try:
        return np.asarray(fitted_baseline.predict(X)).ravel()
    except Exception:
        if split_name == "val" and "yhat_val_base" in globals() and len(yhat_val_base) == len(df):
            return np.asarray(yhat_val_base).ravel()
        if split_name == "test" and "yhat_test_base" in globals() and len(yhat_test_base) == len(df):
            return np.asarray(yhat_test_base).ravel()
        if split_name == "train" and "yhat_train_base" in globals() and len(yhat_train_base) == len(df):
            return np.asarray(yhat_train_base).ravel()
        raise

try:
    _ = BASE_EST.predict(train_df[BASE_FEATS].iloc[:5])
    baseline_model = BASE_EST
except Exception:
    baseline_model = clone(BASE_EST)
    baseline_model.fit(train_df[BASE_FEATS], train_df[TARGET_COL])
    print("Baseline model was refit for this section.")

yhat_val_base  = baseline_predict(val_df, baseline_model, split_name="val")
yhat_test_base = baseline_predict(test_df, baseline_model, split_name="test")


### 7) AR rollout predictions (VAL/TEST)
These are the main outputs: AR model evaluated under strict per-station rollout.


In [ ]:
yhat_val_ar  = rollout_predict(val_df, BASE_EST)
yhat_test_ar = rollout_predict(test_df, BASE_EST)


ValueError: feature_names mismatch: ['precip_mm', 'G_rain_sum_3d', 'G_rain_sum_7d', 'G_rain_sum_30d', 'G_API', 'G_DSLR', 'C_lag_E_SAR_diff_kobs12', 'C_lag_E_SAR_diff_kobs30', 'C_lag_E_SAR_ratio_kobs30', 'C_lag_F_NDVI_kobs30', 'C_lag_LST_modis_kobs12', 'C_lag_LST_modis_kobs30', 'DOY', 'D_sa_E_SAR_ratio', 'D_sa_F_NDMI', 'D_z_E_SAR_ratio', 'D_z_F_NDMI', 'E_SAR_diff', 'E_SAR_ratio', 'F_MSI', 'F_NDMI', 'V_ema_LST_modis_kobs30', 'V_rollmax_E_SAR_diff_kobs14', 'V_rollmax_E_SAR_diff_kobs30', 'V_rollmax_F_NDVI_kobs30', 'V_rollmax_G_API_kobs30', 'V_rollmax_G_API_kobs7', 'V_rollmax_LST_modis_kobs7', 'V_rollmax_s2_b11_kobs30', 'V_rollmean_G_API_kobs30', 'V_rollmin_E_SAR_diff_kobs30', 'V_rollmin_E_SAR_ratio_kobs30', 'V_rollmin_F_NDMI_kobs30', 'V_rollmin_G_API_kobs7', 'V_rollmin_s2_b11_kobs30', 'V_rollmin_s2_b12_kobs30', 's1_vh', 's2_b8', 'A_d_LST_modis_kobs7', 'A_grad_LST_modis_kobs14', 'C_lag_E_SAR_diff_kobs6', 'V_rollmax_E_SAR_diff_kobs7', 'V_rollmax_E_SAR_ratio_kobs14', 'V_rollmax_F_NDVI_kobs14'

### 8) Metrics: baseline vs AR-rollout


In [ ]:
y_val_np  = np.asarray(val_df[TARGET_COL]).ravel()
y_test_np = np.asarray(test_df[TARGET_COL]).ravel()

rows = []
rows.append({"model": "baseline", "split": "val",  **metrics_dict(y_val_np,  yhat_val_base)})
rows.append({"model": "baseline", "split": "test", **metrics_dict(y_test_np, yhat_test_base)})
rows.append({"model": "AR_rollout", "split": "val",  **metrics_dict(y_val_np,  yhat_val_ar)})
rows.append({"model": "AR_rollout", "split": "test", **metrics_dict(y_test_np, yhat_test_ar)})

metrics_df = pd.DataFrame(rows)
display(metrics_df)


### 9) Diagnostics: one station time series + residual histogram
Plots on TEST for a single station.


In [ ]:
sid = test_df[STATION_COL].dropna().unique()[0]
g = test_df[test_df[STATION_COL] == sid].copy()
g[DATE_COL] = pd.to_datetime(g[DATE_COL], errors="coerce")
g = g.sort_values(DATE_COL)

g_base = baseline_predict(g, baseline_model)
g_ar   = rollout_predict(g, ar_model)

plt.figure(figsize=(12, 3.5))
plt.plot(g[DATE_COL], g[TARGET_COL], label="true")
plt.plot(g[DATE_COL], g_base, label="baseline")
plt.plot(g[DATE_COL], g_ar, label="AR_rollout")
plt.title(f"TEST station {sid}: true vs baseline vs AR rollout")
plt.xlabel("date")
plt.ylabel(TARGET_COL)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

res_base = np.asarray(y_test_np) - np.asarray(yhat_test_base)
res_ar   = np.asarray(y_test_np) - np.asarray(yhat_test_ar)

plt.figure(figsize=(12, 3))
plt.hist(res_base, bins=60, alpha=0.6, label="baseline")
plt.hist(res_ar, bins=60, alpha=0.6, label="AR_rollout")
plt.axvline(0)
plt.title("TEST residuals: baseline vs AR rollout")
plt.xlabel("residual (true - pred)")
plt.ylabel("count")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
